## Getting Started with AgentCore, Strands Agents and A2A

[A2A protocol](https://a2a-protocol.org/dev/specification/) is an open standard designed to facilitate communication and interoperability between independent, potentially opaque AI agent systems. In an ecosystem where agents might be built using different frameworks, languages, or by different vendors, A2A provides a common language and interaction model.

[Amazon AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) provides a secure, serverless and purpose-built hosting environment for deploying and running AI agents or tools. 

Recently, AWS announced [A2A support](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-a2a.html) for AgentCore Runtime.

On this workshop, you are going to build following architecture, using AgentCore Runtime:

<img src="images/architecture-lab1.png" style="width: 80%;">

In this first notebook, we are going to build our first agent, an AWS Docs expert. It will query AWS Docs MCP to read and search AWS Documentation and also generate recommendations.

So let's get started!

#### Setup

Install dependencies

In [ ]:
%pip install -q -r requirements.txt --no-cache-dir --force-reinstall

**Please restart your environment, so it can reflect new versions!**

In [ ]:
#import IPython

#IPython.Application.instance().kernel.do_shutdown(True)

Checking if `bedrock-agentcore-starter-toolkit` version is 0.1.21

In [ ]:
!pip freeze | grep boto
!pip freeze | grep agentcore

In [ ]:
# Import libraries
import os
import json
import requests
import boto3
from boto3.session import Session
from strands.tools import tool

# Get boto session
boto_session = Session()
region = boto_session.region_name

### 1 - Create AWS Docs expert Agent
Firstly, let's generate Python code that will be used for our agent, and lately will be deployed in AgentCore.

In [ ]:
%%writefile agents/strands_aws_docs.py
import os
import logging
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.multiagent.a2a import A2AServer
from strands.tools.mcp import MCPClient
from fastapi import FastAPI
import uvicorn


logging.basicConfig(level=logging.INFO)
app = FastAPI()

# Use the complete runtime URL from environment variable, fallback to local
runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')
host, port = "0.0.0.0", 9000



stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

stdio_mcp_client.start()

system_prompt = """You are an AWS Documentation Assistant powered by the AWS Documentation MCP server. Your role is to help users find accurate, up-to-date information from AWS documentation.

Key capabilities:
- Search and retrieve information from AWS service documentation
- Provide clear, accurate answers about AWS services, features, and best practices
- Help users understand AWS concepts, APIs, and configuration options
- Guide users to relevant AWS documentation sections

Guidelines:
- Always prioritize official AWS documentation as your source of truth
- Provide specific, actionable information when possible
- Include relevant links or references to AWS documentation when helpful
- If you're unsure about something, clearly state your limitations
- Focus on being helpful, accurate, and concise in your responses
- Try to simplify/summarize answers to make it faster, small and objective

You have access to AWS documentation search tools to help answer user questions effectively."""

agent = Agent(system_prompt=system_prompt, 
              tools=[stdio_mcp_client.list_tools_sync()],
              name="AWS Docs Agent",
              description="An agent to query AWS Docs using AWS MCP.",
              callback_handler=None)

# Pass runtime_url to http_url parameter AND use serve_at_root=True
a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True  # Serves locally at root (/) regardless of remote URL path complexity
)

@app.get("/ping")
def ping():
    return {"status": "healthy"}

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

#### **Optional** - Local Test

If you want to test this code locally, you can open a bash/terminal window and execute following snippets:

```bash
python agents/strands_aws_docs.py
```

Server will start locally. Then, run in another terminal/bash following commmand to test it:

```bash
curl -X POST http://0.0.0.0:9000 \-H "Content-Type: application/json" \-d '{  "jsonrpc": "2.0",  "id": "req-001",  "method": "message/send",  "params": {  "message": {  "role": "user",  "parts": [  {  "kind": "text",  "text": "O que é o AWS Lambda?"  }  ],  "messageId": "d0673ab9-796d-4270-9435-451912020cd1"  }  } }' | jq .
```

It will query MCP and then return an answer explaining AWS Lambda.

You can also test agent card information retrieval, using following command:

```bash
curl http://localhost:9000/.well-known/agent-card.json | jq .
```

#### Deploy into AgentCore Runtime

Now, let's deploy this solution into AgentCore Runtime.

Let's write a requirements.txt file with dependencies that are needed for the agent.

In [ ]:
%%writefile agents/requirements.txt
strands-agents[a2a]
bedrock-agentcore
strands-agents-tools
pyyaml
ddgs

#### Setup Cognito User Pool

We're going to import a helper class, that has methods to simplify few steps in our workshop.


In [ ]:
from helpers.utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = (
    setup_cognito_user_pool()
)  # You'll get your bearer token from this output cell.
print("Cognito setup completed ✓")

Create IAM Role for Agent

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_DOCS_ROLE_NAME

agent_name="aws_docs_assistant"

execution_role_arn = create_agentcore_runtime_execution_role(AWS_DOCS_ROLE_NAME)

##### Configure server for deployment

We are adding a new protocol in toolkit configuration:

`protocol="A2A"`

This will create this agent to support A2A protocol inside AgentCore

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    entrypoint="agents/strands_aws_docs.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A"
)

print("Configuration completed:", response)

Launch the agent on AgentCore Runtime

In [ ]:
launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

agent_arn = launch_result.agent_arn

**Check Deployment Status**

Let's wait for the deployment to complete:

In [ ]:
import time

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

print(f"Final status: {status}")

Export variables to be used in next notebooks

In [ ]:
AGENT_ID = launch_result.agent_id
AGENT_ARN = launch_result.agent_arn
AGENT_NAME = agent_name
COGNITO_CLIENT_ID = cognito_config.get("client_id")
COGNITO_SECRET = cognito_config.get("client_secret")
DISCOVERY_URL = cognito_config.get("discovery_url")

%store AGENT_ID
%store AGENT_ARN
%store AGENT_NAME
%store COGNITO_CLIENT_ID
%store COGNITO_SECRET
%store DISCOVERY_URL

Store ARN on SSM, so it can be used by orchestrator

In [ ]:
from helpers.utils import put_ssm_parameter, SSM_DOCS_AGENT_ARN

put_ssm_parameter(SSM_DOCS_AGENT_ARN, AGENT_ARN)

#### Invoking A2A agent

Let's start by getting Agent Card information, after refreshing the auth token:

In [ ]:
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"), 
    cognito_config.get("client_secret")
)

In [ ]:
from uuid import uuid4
from urllib.parse import quote


session_id = str(uuid4())
print(f"Generated session ID: {session_id}") # temp keeping same session

def fetch_agent_card(session_id):
    # URL encode the agent ARN
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/.well-known/agent-card.json"
    print(url)
    # Generate a unique session ID
    #session_id = str(uuid4())
    #print(f"Generated session ID: {session_id}")

    # Set headers
    headers = {
        'Accept': '*/*',
        'Authorization': f'Bearer {bearer_token}',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
        'X-Amzn-Trace-Id': f'aws_docs_assistant_{session_id}'
    }

    try:
        # Make the request
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parse and pretty print JSON
        agent_card = response.json()
        print(json.dumps(agent_card, indent=2))

        return agent_card

    except requests.exceptions.RequestException as e:
        print(f"Error fetching agent card: {e}")
        return None

In [ ]:
fetch_agent_card(session_id)

Now, let's invoke the agent, using A2A:

In [ ]:
import asyncio
import logging
import os
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 300  # set request timeout to 5 minutes

def create_message(*, role: Role = Role.user, text: str) -> Message:
    return Message(
        kind="message",
        role=role,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )

async def send_sync_message(message: str):
    # Get runtime URL from environment variable
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/"
    
    # Generate a unique session ID
    session_id = str(uuid4())
    print(f"Generated session ID: {session_id}")

    # Add authentication headers for AgentCore
    headers = {"Authorization": f"Bearer {bearer_token}",
              'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id}
        
    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, headers=headers) as httpx_client:
        # Get agent card from the runtime URL
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
        agent_card = await resolver.get_agent_card()
        print(agent_card)

        # Agent card contains the correct URL (same as runtime_url in this case)
        # No manual override needed - this is the path-based mounting pattern

        # Create client using factory
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=False,  # Use non-streaming mode for sync response
        )
        factory = ClientFactory(config)
        client = factory.create(agent_card)

        # Create and send message
        msg = create_message(text=message)

        # With streaming=False, this will yield exactly one result
        async for event in client.send_message(msg):
            if isinstance(event, Message):
                logger.info(event.model_dump_json(exclude_none=True, indent=2))
                return event
            elif isinstance(event, tuple) and len(event) == 2:
                # (Task, UpdateEvent) tuple
                task, update_event = event
                logger.info(f"Task: {task.model_dump_json(exclude_none=True, indent=2)}")
                if update_event:
                    logger.info(f"Update: {update_event.model_dump_json(exclude_none=True, indent=2)}")
                return task
            else:
                # Fallback for other response types
                logger.info(f"Response: {str(event)}")
                return event

In [ ]:
result = await send_sync_message("what is DynamoDB")

Congratulations, you have deployed your first agent, using A2A protocol on Amazon AgentCore Runtime.

Now, let's move to next lab.